In [1]:
%pip install qdrant-client sentence-transformers numpy

   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.5/4.3 MB 2.9 MB/s eta 0:00:02
   ---- ----------------------------------- 0.5/4.3 MB 2.9 MB/s eta 0:00:02
   ------- -------------------------------- 0.8/4.3 MB 1.5 MB/s eta 0:00:03
   ------------ --------------------------- 1.3/4.3 MB 1.5 MB/s eta 0:00:02
   -------------- ------------------------- 1.6/4.3 MB 1.5 MB/s eta 0:00:02
   ----------------- ---------------------- 1.8/4.3 MB 1.4 MB/s eta 0:00:02
   ------------------- -------------------- 2.1/4.3 MB 1.4 MB/s eta 0:00:02
   ---------------------- ----------------- 2.4/4.3 MB 1.4 MB/s eta 0:00:02
   ------------------------ --------------- 2.6/4.3 MB 1.4 MB/s eta 0:00:02
   -------------------------- ------------- 2.9/4.3 MB 1.4 MB/s eta 0:00:02
   ----------------------------- ---------- 3.1/4.3 MB 1.4 MB/s eta 0:00:01
   -----------------------

In [ ]:
import numpy as np
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import random
from typing import List, Dict, Optional, Tuple

class SkipNode:
    
    def __init__(self, value: str, level: int):
        self.value = value
        self.forward = [None] * (level + 1)

class SkipList:
    
    def __init__(self, max_level: int = 16, p: float = 0.5):
        self.max_level = max_level
        self.p = p
        self.header = SkipNode(None, max_level)
        self.level = 0

    def random_level(self) -> int:
        
        level = 0
        while random.random() < self.p and level < self.max_level - 1:
            level += 1
        return level

    def insert(self, value: str) -> None:
        
        update = [self.header] * self.max_level  
        current = self.header
        for i in range(self.level - 1, -1, -1):
            while current.forward[i] and current.forward[i].value < value:
                current = current.forward[i]
            update[i] = current
        level = self.random_level()
        if level > self.level:
            for i in range(self.level, level + 1):
                update[i] = self.header
            self.level = level + 1
        new_node = SkipNode(value, level)
        for i in range(level + 1):
            new_node.forward[i] = update[i].forward[i]
            update[i].forward[i] = new_node

    def search(self, value: str) -> Optional[str]:
        
        current = self.header
        for i in range(self.level - 1, -1, -1):
            while current.forward[i] and current.forward[i].value < value:
                current = current.forward[i]
        current = current.forward[0]
        if current and current.value == value:
            return current.value
        return None

class EmployeeSearchSystem:
    
    def __init__(self):
        self.qdrant = QdrantClient(":memory:")  
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.skip_lists: Dict[Tuple[str, str], SkipList] = {}
        self.vector_size = self.model.get_sentence_embedding_dimension() + 2  

    def create_employee_vector(self, employee: Dict[str, str]) -> np.ndarray:
        
        name_embedding = self.model.encode(employee["name"])
        company_code = hash(employee["company"]) % 100 / 100.0  
        department_code = hash(employee["department"]) % 100 / 100.0
        return np.concatenate([name_embedding, [company_code, department_code]])

    def initialize_collection(self) -> None:
        
        try:
            self.qdrant.recreate_collection(
                collection_name="employees",
                vectors_config=models.VectorParams(
                    size=self.vector_size,
                    distance=models.Distance.COSINE
                ),
                hnsw_config=models.HnswConfigDiff(
                    m=16,
                    ef_construct=100,
                    full_scan_threshold=10000
                )
            )
        except Exception as e:
            print(f"Error initializing collection: {e}")
            self.qdrant.recreate_collection(
                collection_name="employees",
                vectors_config=models.VectorParams(
                    size=self.vector_size,
                    distance=models.Distance.COSINE
                ),
                hnsw_config=models.HnswConfigDiff(
                    m=16,
                    full_scan_threshold=10000
                )
            )

    def index_employees(self, employees: List[Dict[str, str]]) -> None:
        
        vectors = [self.create_employee_vector(emp) for emp in employees]
        payloads = [{"id": emp["id"], "name": emp["name"], "company": emp["company"], 
                     "department": emp["department"]} for emp in employees]
        ids = [i for i in range(len(employees))]
        self.qdrant.upload_collection(collection_name="employees", vectors=vectors, payload=payloads, ids=ids)

        for emp in employees:
            key = (emp["company"], emp["department"])
            if key not in self.skip_lists:
                self.skip_lists[key] = SkipList()
            self.skip_lists[key].insert(emp["id"])

    def find_employee(self, name: str, company: str, department: str, 
                     employee_id: Optional[str] = None) -> Optional[Dict[str, str]]:
        
        query_vector = self.create_employee_vector({"name": name, "company": company, "department": department})
        results = self.qdrant.search(
            collection_name="employees",
            query_vector=query_vector,
            limit=5,
            with_payload=True
        )
        key = (company, department)
        if key in self.skip_lists:
            for result in results:
                emp_id = result.payload["id"]
                if self.skip_lists[key].search(emp_id):
                    if employee_id and emp_id == employee_id:
                        return result.payload
                    elif not employee_id:
                        print(f"Candidate: {result.payload}")
            return None
        return None


if __name__ == "__main__":
    employees = [
        {"id": "12345", "name": "John Doe", "company": "Tech Corp", "department": "Engineering"},
        {"id": "78906", "name": "John Doe", "company": "Tech Corp", "department": "Sales"},
        {"id": "54321", "name": "Jane Smith", "company": "Data Inc", "department": "Engineering"},
    ]

    system = EmployeeSearchSystem()
    system.initialize_collection()
    system.index_employees(employees)

    result = system.find_employee("John Doe", "Tech Corp", "Engineering", "12345")
    if result:
        print(f"Found: {result}")
    else:
        print("Employee not found")

C:\Users\Hello!\AppData\Local\Temp\ipykernel_11532\2211313836.py:75: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  self.qdrant.recreate_collection(


Employee not found


C:\Users\Hello!\AppData\Local\Temp\ipykernel_11532\2211313836.py:119: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = self.qdrant.search(
